# Final 4-stage BSD35k filtered datasets

This notebook creates four BSD35k candidate datasets for team handoff.

- A: `v4_ge4` as the safest validated dataset.
- B: `v4_ge3` as a wider coverage candidate.
- C: `v4_ge4 + sp-p` expansion.
- D: `v4_ge4 + sp-p + sp-c` probe expansion.

Expansion rows are selected from BSD35k rows not already in `v4_ge4`, using v3 high-confidence probability and label-quality filtering. Class caps are based on BSD10k train_pool class counts.

In [1]:
from pathlib import Path
import json
import math

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'baseline_confidnce_train').exists():
    ROOT = ROOT.parent

V4_DATA_DIR = ROOT / 'baseline_confidnce_train' / 'outputs' / 'v4_35k_hier_loss' / 'datasets'
SPLIT_PATH = ROOT / 'baseline_confidnce_train' / 'outputs' / 'v4_35k_hier_loss' / 'fixed_train_pool_final_test_split.csv'
V3_PATH = ROOT / 'outputs' / 'confidence_filter_v3' / 'predictions' / 'BSD35k-CS_filter_predictions.csv'
LABEL_QUALITY_PATH = ROOT / 'experiments' / 'bsd35k_label_quality' / 'BSD35k-CS_label_quality_scores.csv'

OUTPUT_DIR = ROOT / 'baseline_confidnce_train' / 'outputs' / 'final_4stage_filtered_datasets'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Conservative defaults. v3 0.40 is the F1-optimal threshold from the binary filter.
# Raising this to 0.70 is too strict for sp-p/sp-c expansion after removing v4_ge4 rows.
V3_PROB_THRESHOLD = 0.40
LABEL_QUALITY_THRESHOLD = 0.35
SP_P_CAP_RATIO = 1.00
SP_C_CAP_RATIO = 0.50  # use 0.25 for a stricter probe

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

ROOT: c:\Users\solok\Desktop\Dcase baseline
OUTPUT_DIR: c:\Users\solok\Desktop\Dcase baseline\baseline_confidnce_train\outputs\final_4stage_filtered_datasets


## 1. Load source tables

In [2]:
all_usable = pd.read_csv(V4_DATA_DIR / 'bsd35k_all_usable.csv')
v4_ge3 = pd.read_csv(V4_DATA_DIR / 'bsd35k_v4_ge3.csv')
v4_ge4 = pd.read_csv(V4_DATA_DIR / 'bsd35k_v4_ge4.csv')
v3 = pd.read_csv(V3_PATH)
label_quality = pd.read_csv(LABEL_QUALITY_PATH)
split_df = pd.read_csv(SPLIT_PATH)

for df in [all_usable, v4_ge3, v4_ge4, v3, label_quality, split_df]:
    if 'sound_id' in df.columns:
        df['sound_id'] = df['sound_id'].astype(str)
    if 'index' in df.columns:
        df['index'] = df['index'].astype(str)

train_pool = split_df[split_df['split'].eq('train_pool')].copy()
train_class_counts = train_pool['class'].value_counts().to_dict()

print('all usable:', len(all_usable))
print('v4_ge3:', len(v4_ge3))
print('v4_ge4:', len(v4_ge4))
print('train classes:', train_pool['class'].nunique())
print('sp-p train count:', train_class_counts.get('sp-p', 0))
print('sp-c train count:', train_class_counts.get('sp-c', 0))

all usable: 31464
v4_ge3: 15566
v4_ge4: 7633
train classes: 23
sp-p train count: 289
sp-c train count: 140


## 2. Merge v3 and label-quality scores

In [3]:
v3_cols = [
    'sound_id',
    'predicted_high_confidence_prob',
    'predicted_high_confidence',
    'predicted_high_confidence_strict',
    'binary_f1_optimal_threshold',
    'binary_precision_optimal_recall70_threshold',
]
lq_cols = [
    'sound_id',
    'label_quality_score',
    'label_noise_score',
    'quality_group',
    'same_class',
    'same_top_class',
    'provided_class_probability',
    'classifier_margin',
    'recommended_action',
]

score_df = all_usable.merge(v3[v3_cols], on='sound_id', how='left')
score_df = score_df.merge(label_quality[lq_cols], on='sound_id', how='left')

missing_v3 = score_df['predicted_high_confidence_prob'].isna().sum()
missing_lq = score_df['label_quality_score'].isna().sum()
print('missing v3 scores:', missing_v3)
print('missing label quality:', missing_lq)

score_df.head()

missing v3 scores: 0
missing label quality: 0


,sound_id,class,top_class,v4_filter_score,binary_mlp_prob,fiveclass_score,fiveclass_p45,index,class_idx,top_class_idx,...,binary_f1_optimal_threshold,binary_precision_optimal_recall70_threshold,label_quality_score,label_noise_score,quality_group,same_class,same_top_class,provided_class_probability,classifier_margin,recommended_action
0,796088,fx-h,fx,0.851735,0.745651,3.783054,0.784744,796088,14,3,...,0.4,0.585,0.923710,0.076290,clean_high_quality,True,True,0.912587,0.890894,use_high_weight
1,796133,fx-n,fx,0.384630,0.501452,3.406448,0.441924,796133,16,3,...,0.4,0.585,0.190544,0.809456,middle_uncertain,False,True,0.017158,0.197508,use_low_weight_or_review
2,796149,ss-n,ss,0.327088,0.471124,3.375044,0.448676,796149,19,4,...,0.4,0.585,0.318939,0.681061,ambiguous_same_top,False,True,0.101223,0.745066,use_hierarchy_soft_label_or_low_weight
3,796181,ss-n,ss,0.661121,0.588156,3.671201,0.702746,796181,19,4,...,0.4,0.585,0.961307,0.038693,clean_high_quality,True,True,0.966446,0.946484,use_high_weight
4,796221,is-e,is,0.069842,0.357983,2.913855,0.241068,796221,7,1,...,0.4,0.585,0.203911,0.796089,middle_uncertain,False,False,0.216226,0.432697,use_low_weight_or_review


## 3. Select controlled expansion rows

In [4]:
v4_ge4_ids = set(v4_ge4['sound_id'].astype(str))

def select_class_expansion(class_name: str, cap_ratio: float) -> pd.DataFrame:
    cap = math.floor(train_class_counts.get(class_name, 0) * cap_ratio)
    candidates = score_df[
        (~score_df['sound_id'].isin(v4_ge4_ids))
        & score_df['class'].eq(class_name)
        & score_df['predicted_high_confidence_prob'].ge(V3_PROB_THRESHOLD)
        & score_df['label_quality_score'].ge(LABEL_QUALITY_THRESHOLD)
    ].copy()
    candidates = candidates.sort_values(
        ['predicted_high_confidence_prob', 'label_quality_score', 'v4_filter_score'],
        ascending=False,
    )
    selected = candidates.head(cap).copy()
    selected['expansion_class_cap_ratio'] = cap_ratio
    selected['expansion_class_cap'] = cap
    selected['expansion_rank_within_class'] = range(1, len(selected) + 1)
    print(
        class_name,
        'cap_ratio=', cap_ratio,
        'cap=', cap,
        'candidates=', len(candidates),
        'selected=', len(selected),
    )
    return selected

sp_p_extra = select_class_expansion('sp-p', SP_P_CAP_RATIO)
sp_c_extra = select_class_expansion('sp-c', SP_C_CAP_RATIO)

display(sp_p_extra[['sound_id', 'class', 'predicted_high_confidence_prob', 'label_quality_score', 'v4_filter_score']].head())
display(sp_c_extra[['sound_id', 'class', 'predicted_high_confidence_prob', 'label_quality_score', 'v4_filter_score']].head())

sp-p cap_ratio= 1.0 cap= 289 candidates= 16 selected= 16
sp-c cap_ratio= 0.5 cap= 70 candidates= 10 selected= 10


,sound_id,class,predicted_high_confidence_prob,label_quality_score,v4_filter_score
26140,835836,sp-p,0.624375,0.675798,0.600035
15513,819318,sp-p,0.618911,0.952002,0.578216
11142,813121,sp-p,0.596101,0.898202,0.604500
26058,835738,sp-p,0.574576,0.474164,0.287964
19376,826407,sp-p,0.539213,0.628378,0.555556


,sound_id,class,predicted_high_confidence_prob,label_quality_score,v4_filter_score
22074,830493,sp-c,0.530742,0.830154,0.508804
10928,812791,sp-c,0.517939,0.763435,0.393926
16425,822633,sp-c,0.501001,0.839624,0.408165
24746,834033,sp-c,0.452287,0.625158,0.280177
4587,803834,sp-c,0.441967,0.565384,0.268132


## 4. Build the four datasets

In [5]:
DCASE_COLS = [
    'sound_id', 'class', 'top_class', 'v4_filter_score', 'binary_mlp_prob',
    'fiveclass_score', 'fiveclass_p45', 'index', 'class_idx', 'top_class_idx',
    'audio_emb_filepath', 'text_emb_filepath', 'predicted_confidence_score',
]

def attach_scores(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    out = df.copy()
    out['sound_id'] = out['sound_id'].astype(str)
    if 'predicted_high_confidence_prob' not in out.columns:
        extra_cols = ['sound_id', 'predicted_high_confidence_prob', 'label_quality_score', 'label_noise_score', 'quality_group', 'recommended_action']
        out = out.merge(score_df[extra_cols], on='sound_id', how='left')
    out['dataset_source'] = source_name
    return out

def dedupe_by_sound_id(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates('sound_id', keep='first').reset_index(drop=True)

dataset_a = attach_scores(v4_ge4, 'A_v4_ge4')
dataset_b = attach_scores(v4_ge3, 'B_v4_ge3')
dataset_c = dedupe_by_sound_id(pd.concat([attach_scores(v4_ge4, 'base_v4_ge4'), sp_p_extra], ignore_index=True))
dataset_c['dataset_source'] = dataset_c['dataset_source'].fillna('C_sp_p_extra')
dataset_d = dedupe_by_sound_id(pd.concat([attach_scores(v4_ge4, 'base_v4_ge4'), sp_p_extra, sp_c_extra], ignore_index=True))
dataset_d['dataset_source'] = dataset_d['dataset_source'].fillna('D_sp_p_or_sp_c_extra')

datasets = {
    'A_v4_ge4': dataset_a,
    'B_v4_ge3': dataset_b,
    'C_v4_ge4_plus_sp_p': dataset_c,
    'D_v4_ge4_plus_sp_p_sp_c_probe': dataset_d,
}

for name, df in datasets.items():
    print(name, len(df), df['class'].nunique())
    display(df['class'].value_counts().rename('count').head(10).to_frame())

A_v4_ge4 7633 23


,count
class,
fx-o,1332
is-p,837
m-m,730
m-si,717
fx-v,706
sp-s,600
m-sp,555
fx-m,353
fx-h,301


B_v4_ge3 15566 23


,count
class,
fx-o,2713
fx-v,1516
m-m,1331
is-p,1228
ss-n,1062
m-si,973
fx-m,883
fx-el,836
fx-h,754


C_v4_ge4_plus_sp_p 7649 23


,count
class,
fx-o,1332
is-p,837
m-m,730
m-si,717
fx-v,706
sp-s,600
m-sp,555
fx-m,353
fx-h,301


D_v4_ge4_plus_sp_p_sp_c_probe 7659 23


,count
class,
fx-o,1332
is-p,837
m-m,730
m-si,717
fx-v,706
sp-s,600
m-sp,555
fx-m,353
fx-h,301


## 5. Save outputs

In [6]:
manifest_rows = []

for name, df in datasets.items():
    full_path = OUTPUT_DIR / f'{name}_with_scores.csv'
    dcase_path = OUTPUT_DIR / f'{name}_dcase_rows.csv'
    counts_path = OUTPUT_DIR / f'{name}_class_counts.csv'

    df.to_csv(full_path, index=False)
    df[DCASE_COLS].to_csv(dcase_path, index=False)
    df['class'].value_counts().sort_index().rename('count').to_csv(counts_path)

    manifest_rows.append({
        'dataset_name': name,
        'rows': len(df),
        'classes': df['class'].nunique(),
        'sp_p_rows': int(df['class'].eq('sp-p').sum()),
        'sp_c_rows': int(df['class'].eq('sp-c').sum()),
        'full_with_scores_csv': str(full_path),
        'dcase_rows_csv': str(dcase_path),
        'class_counts_csv': str(counts_path),
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR / 'dataset_manifest.csv', index=False)

config = {
    'v3_prob_threshold': V3_PROB_THRESHOLD,
    'label_quality_threshold': LABEL_QUALITY_THRESHOLD,
    'sp_p_cap_ratio': SP_P_CAP_RATIO,
    'sp_c_cap_ratio': SP_C_CAP_RATIO,
    'sp_p_train_count': int(train_class_counts.get('sp-p', 0)),
    'sp_c_train_count': int(train_class_counts.get('sp-c', 0)),
    'output_dir': str(OUTPUT_DIR),
}
(OUTPUT_DIR / 'selection_config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')

display(manifest)
print('saved to:', OUTPUT_DIR)

,dataset_name,rows,classes,sp_p_rows,sp_c_rows,full_with_scores_csv,dcase_rows_csv,class_counts_csv
0,A_v4_ge4,7633,23,7,2,c:\Users\solok\Desktop\Dcase baseline\baseline...,c:\Users\solok\Desktop\Dcase baseline\baseline...,c:\Users\solok\Desktop\Dcase baseline\baseline...
1,B_v4_ge3,15566,23,15,6,c:\Users\solok\Desktop\Dcase baseline\baseline...,c:\Users\solok\Desktop\Dcase baseline\baseline...,c:\Users\solok\Desktop\Dcase baseline\baseline...
2,C_v4_ge4_plus_sp_p,7649,23,23,2,c:\Users\solok\Desktop\Dcase baseline\baseline...,c:\Users\solok\Desktop\Dcase baseline\baseline...,c:\Users\solok\Desktop\Dcase baseline\baseline...
3,D_v4_ge4_plus_sp_p_sp_c_probe,7659,23,23,12,c:\Users\solok\Desktop\Dcase baseline\baseline...,c:\Users\solok\Desktop\Dcase baseline\baseline...,c:\Users\solok\Desktop\Dcase baseline\baseline...


saved to: c:\Users\solok\Desktop\Dcase baseline\baseline_confidnce_train\outputs\final_4stage_filtered_datasets
